# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
# Optional: Explore all top-level metadata attributes
for attr in dir(metadata):
    if not attr.startswith("_") and not callable(getattr(metadata, attr)):
        print(f"{attr}: {getattr(metadata, attr)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List all Record Sets by @id
print("Record Sets in the dataset:")
record_sets = []
for record_set in metadata.record_sets:
    print(f"  • id: {record_set.id} | name: {record_set.name}")
    record_sets.append(record_set.id)

# For this dataset, get an overview of the first Record Set
if record_sets:
    selected_rs_id = record_sets[0]
    rs = dataset.get_record_set(record_set=selected_rs_id)
    print(f"\nFields for Record Set {selected_rs_id}:")
    for field in rs.fields:
        print(f"  • id: {field.id} | name: {field.name} | dataType: {field.data_type}")
else:
    print('No record sets found. Please check the Croissant schema or the dataset definition.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    print(f"\nLoading record set: {record_set_id}")
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    print(f"Fields: {list(df.columns)}\nRows: {len(df)}\n")
    dataframes[record_set_id] = df

# Display columns (fields) of the first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"Columns (fields) for record set {first_rs}:\n", dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Pick first record set
if record_sets:
    rs_id = record_sets[0]
    df = dataframes[rs_id]
    # Guess a numeric field by finding columns with numeric dtype
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_field_candidates:
        # Try to coerce possible numeric fields (sometimes they are object dtype)
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                numeric_field_candidates.append(col)
            except Exception:
                pass

    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field}\n")
        # Convert if necessary
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Example: Filter values > threshold (use median for demonstration)
        threshold = df[numeric_field].median() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}, n={len(filtered_df)}\n")

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by first non-numeric field, if available
        group_field_candidates = [col for col in df.columns if col != numeric_field and pd.api.types.is_string_dtype(df[col])]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by '{group_field}':\n", grouped_df.head())
        else:
            print("No suitable categorical field found for groupby.")
    else:
        print('No numeric field found in the first record set for analysis. Please check field types.')
else:
    print('No record sets available to analyze.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Example: Histogram and Boxplot
if record_sets and 'numeric_field' in locals():
    # Drop NA for plotting
    series = df[numeric_field].dropna()
    if not series.empty:
        plt.figure(figsize=(12,5))
        plt.subplot(1,2,1)
        plt.hist(series, bins=30, color='skyblue', edgecolor='black')
        plt.xlabel(numeric_field)
        plt.title(f"Distribution of {numeric_field}")

        plt.subplot(1,2,2)
        plt.boxplot(series, vert=False)
        plt.xlabel(numeric_field)
        plt.title(f"Boxplot of {numeric_field}")
        plt.tight_layout()
        plt.show()
    else:
        print(f"No data available in field '{numeric_field}' for visualization.")
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library. We discovered available record sets using their `@id` fields, loaded tabular data into Pandas DataFrames, and performed simple exploratory analysis and visualization on the identified numeric fields. Further domain-specific analysis and deeper modeling could be performed as needed.